In [0]:
# Step 1: Set up the Bronze and Silver tables
from pyspark.sql import functions as F
from delta.tables import DeltaTable

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema_name = "retail_lakehouse"

bronze_table = f"{catalog}.{schema_name}.streaming_orders_bronze"
silver_table = f"{catalog}.{schema_name}.streaming_orders_silver"

print("Bronze source:", bronze_table)
print("Silver target:", silver_table)

In [0]:
# Step 2: Read the orders captured by Structured Streaming
bronze_df = spark.table(bronze_table)

print("Bronze row count:", bronze_df.count())
display(bronze_df)

In [0]:
# Step 3: Prepare the records for Silver
prepared_df = (
    bronze_df
    .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))
    .dropDuplicates(["order_id"])
)

valid_condition = (
    F.col("order_id").isNotNull()
    & F.col("order_date").isNotNull()
    & F.col("product_id").isNotNull()
    & F.col("quantity").isNotNull()
    & (F.col("quantity") > 0)
    & F.col("unit_price").isNotNull()
    & (F.col("unit_price") >= 0)
)

valid_orders = prepared_df.filter(valid_condition)
invalid_orders = prepared_df.filter(~valid_condition)

print("Valid records:", valid_orders.count())
print("Invalid records:", invalid_orders.count())

display(valid_orders)

In [0]:
# Step 4: Create an empty Silver table with the cleaned schema
if not spark.catalog.tableExists(silver_table):
    (
        valid_orders.limit(0)
        .write
        .format("delta")
        .saveAsTable(silver_table)
    )

print("Silver table is ready.")

In [0]:
# Step 5: Insert only new order IDs into Silver
silver_delta = DeltaTable.forName(spark, silver_table)

(
    silver_delta.alias("target")
    .merge(
        valid_orders.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver merge completed.")

In [0]:
# Step 6: Check the cleaned records
silver_df = spark.table(silver_table)

print("Silver row count:", silver_df.count())
display(silver_df.orderBy("order_date", "order_id"))

In [0]:
# Step 7: Repeat the merge to check for duplicate inserts
silver_delta = DeltaTable.forName(spark, silver_table)

(
    silver_delta.alias("target")
    .merge(
        valid_orders.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver rows after rerun:", spark.table(silver_table).count())